# Experimento 01 · Bases de comparación (E0) y residuo sobre GFS (E1)

Objetivo: medir qué tanto aporta cada idea antes de pasar a modelos más grandes.

- **E0 · Bases de comparación:** climatología, GFS interpolado a 3 km, interpolación con corrección de temperatura por altura, regresión lineal por celda y el CNN del starter.
- **E1 · Residuo sobre GFS:** una red convolucional que trabaja en la grilla de 3 km y predice solo la corrección `WRF − GFS interpolado`, con relieve como entrada y una pérdida alineada con la métrica.

Validación principal: igual que el starter (desde el 16 de junio). Para las bases lineales se agregan cortes temporales adicionales.
Variable de entorno `QUICK=1` para una corrida corta de prueba. `NTHREADS` limita los hilos de torch. `LOSS=rmse` activa la pérdida alineada con la métrica (por defecto, MSE normalizada).

Protocolo: número de épocas fijo y se reporta la última (sin elegir época con la validación). El modelo que se guarda y se usa para el submission se reentrena con todos los datos de entrenamiento.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
torch.set_num_threads(int(os.environ.get('NTHREADS', os.cpu_count() or 1)))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
QUICK = os.environ.get('QUICK') == '1'
TAG = '_quick' if QUICK else ''        # evita mezclar salidas de prueba con las completas
EP_STARTER, EP_E1 = (2, 2) if QUICK else (25, 20)
print('dispositivo:', dev, '| hilos:', torch.get_num_threads(), '| QUICK:', QUICK)

def find(name):
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    print('usando', hits[0])
    return hits[0]

tr, te, st = np.load(find('train.npz')), np.load(find('test.npz')), np.load(find('static.npz'))
Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem'].astype(np.float32)
VARS = ['RH', 'T', 'U', 'V']
REF = np.array([25.360, 2.0652, 2.6776, 2.0876])   # RMSE de la climatología según la página de evaluación
print(Xtr.shape, Ytr.shape, Xte.shape, dem.shape)

## Utilidades: horas, ventanas, validación y métricas

In [ ]:
hour = lambda t: np.array([int(k[-2:]) for k in t])

def windows(X, t):
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        i1 = pos.get(k - pd.Timedelta(hours=1), None)
        i2 = pos.get(k - pd.Timedelta(hours=2), None)
        # relleno: si falta un retraso se usa el disponible más cercano a t (t-1 antes que t)
        out[i, 2] = X[i]
        out[i, 1] = X[i1] if i1 is not None else X[i]
        out[i, 0] = X[i2] if i2 is not None else out[i, 1]
    return out

Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)
val = ttr >= '2024061600'
trn = ~val

def climatology(Y, t):
    h = hour(t)
    return np.stack([Y[h == k].mean(0) for k in range(24)])

clim_trn = climatology(Ytr[trn], ttr[trn])
clim_rmse_val = np.sqrt(((clim_trn[hour(ttr[val])] - Ytr[val]) ** 2).mean(axis=(0, 1, 2)))

def scores(pred, true, clim_rmse=clim_rmse_val):
    rmse = np.sqrt(((pred - true) ** 2).mean(axis=(0, 1, 2)))
    return {'score_local': 1 - (rmse / clim_rmse).mean(), 'score_ref': 1 - (rmse / REF).mean(),
            **{f'RMSE_{v}': r for v, r in zip(VARS, rmse)}}

RESULTS = []
def log(name, pred, true=None):
    s = scores(pred, Ytr[val] if true is None else true)
    if true is None and len(pred) == val.sum():      # el leaderboard solo evalúa 00, 03, ..., 21 UTC
        m3 = hour(ttr[val]) % 3 == 0
        s['score_ref_3h'] = scores(pred[m3], Ytr[val][m3])['score_ref']
    RESULTS.append({'experimento': name, **s})
    print(f"{name:42s} score_local {s['score_local']:.4f} | score_ref {s['score_ref']:.4f} | ref_3h {s.get('score_ref_3h', float('nan')):.4f} | " +
          ' '.join(f"{v} {s['RMSE_'+v]:.3f}" for v in VARS))
    return s

print('validación:', val.sum(), 'h | entrenamiento:', trn.sum(), 'h')
print('RMSE climatología en validación:', dict(zip(VARS, clim_rmse_val.round(3))), '| referencia oficial:', dict(zip(VARS, REF)))

## Interpolación del GFS a la grilla de 3 km y derivados del relieve

In [ ]:
glat, glon = st['gfs_lats'], st['gfs_lons']     # GFS: fila 0 = norte
wlat, wlon = st['wrf_lats'], st['wrf_lons']     # WRF: fila 0 = sur
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32, device=dev)   # (1,70,35,2)

def interp(x):
    # x: tensor (B, C, 17, 13) -> (B, C, 70, 35), bilineal según coordenadas reales
    return F.grid_sample(x, GRID.expand(x.shape[0], -1, -1, -1), mode='bilinear', align_corners=True)

def interp_np(X, bs=512):
    out = []
    with torch.no_grad():
        for s in range(0, len(X), bs):
            x = torch.tensor(X[s:s+bs], device=dev).permute(0, 3, 1, 2)
            out.append(interp(x).permute(0, 2, 3, 1).cpu().numpy())
    return np.concatenate(out)

Xi_tr = interp_np(Xtr)            # GFS de la hora t interpolado: (N, 70, 35, 4)

# Relieve suavizado a ~27 km (ventana de 9 x 9 celdas de 3 km) y diferencia con el relieve real
k = 9
dem_t = torch.tensor(dem)[None, None]
dem_s = F.avg_pool2d(F.pad(dem_t, (k//2,)*4, mode='replicate'), k, stride=1)[0, 0].numpy()
dz = dem - dem_s
gyy, gxx = np.gradient(dem, 3000.0)
slope = np.hypot(gxx, gyy)
print('dz (m): min %.0f, max %.0f | pendiente máx %.3f' % (dz.min(), dz.max(), slope.max()))

## E0 · Bases de comparación

In [ ]:
log('E0a climatología', clim_trn[hour(ttr[val])])
log('E0b GFS interpolado', Xi_tr[val])

# Corrección de temperatura por altura: T = T_interp + gamma * dz, gamma ajustado en entrenamiento
resT = (Ytr[trn, ..., 1] - Xi_tr[trn, ..., 1]).reshape(len(Ytr[trn]), -1)
gamma = float((resT * dz.reshape(1, -1)).sum() / ((dz ** 2).sum() * len(resT)))
pred = Xi_tr[val].copy(); pred[..., 1] += gamma * dz
print('gamma ajustado: %.2f K/km' % (gamma * 1000))
log('E0c interpolado + corrección T por altura', pred)

# Regresión lineal por celda (ridge pequeño): rasgos = [1, GFS interpolado t y t-1 (8), sin/cos hora]
Wi_prev = interp_np(Wtr[:, 1])
def feats(idx):
    h = hour(ttr[idx]); ang = 2 * np.pi * h / 24
    n = idx.sum() if idx.dtype == bool else len(idx)
    hh = np.stack([np.sin(ang), np.cos(ang)], 1)[:, None, None, :].repeat(70, 1).repeat(35, 2)
    return np.concatenate([np.ones((n, 70, 35, 1), np.float32), Xi_tr[idx], Wi_prev[idx], hh], -1).astype(np.float64)

def fit_pixel_linear(idx_fit, lam=1e-3):
    A, Y = feats(idx_fit), Ytr[idx_fit].astype(np.float64)
    mu, sd = A.mean(0, keepdims=True), A.std(0, keepdims=True) + 1e-6
    mu[..., 0], sd[..., 0] = 0, 1
    n, nf = len(A), A.shape[-1]
    An = ((A - mu) / sd).reshape(n, -1, nf).transpose(1, 0, 2)      # (celdas, n, rasgos)
    Yp = Y.reshape(n, -1, 4).transpose(1, 0, 2)                      # (celdas, n, 4)
    G = An.transpose(0, 2, 1) @ An + lam * n * np.eye(nf)
    G[:, 0, 0] -= lam * n                                            # el intercepto no se penaliza
    W = np.linalg.solve(G, An.transpose(0, 2, 1) @ Yp)              # (celdas, rasgos, 4)
    def predict(idx):
        B = ((feats(idx) - mu) / sd); m = len(B)
        return (B.reshape(m, -1, nf).transpose(1, 0, 2) @ W).transpose(1, 0, 2).reshape(m, 70, 35, 4).astype(np.float32)
    return predict

lin = fit_pixel_linear(trn)
pred_lin = lin(val)
log('E0d regresión lineal por celda', pred_lin)

### Cortes temporales adicionales (bases lineales)
Se entrena solo con datos anteriores a cada corte y se valida en las dos semanas siguientes. Sirve para ver cuánto varía el score entre periodos.

In [ ]:
cuts = [('2024-05-01', '2024-05-15'), ('2024-05-16', '2024-05-31'), ('2024-06-01', '2024-06-15'), ('2024-06-16', '2024-06-30')]
rows = []
for a, b in cuts:
    A, B = a.replace('-', '') + '00', b.replace('-', '') + '23'
    fit, ev = ttr < A, (ttr >= A) & (ttr <= B)
    cl = climatology(Ytr[fit], ttr[fit])
    cr = np.sqrt(((cl[hour(ttr[ev])] - Ytr[ev]) ** 2).mean(axis=(0, 1, 2)))
    f = fit_pixel_linear(fit)
    for name, p in [('GFS interpolado', Xi_tr[ev]), ('regresión lineal por celda', f(ev))]:
        s = scores(p, Ytr[ev], cr)
        rows.append({'corte': f'{a} a {b}', 'modelo': name, 'horas_ajuste': int(fit.sum()), **{k: round(float(v), 4) for k, v in s.items()}})
folds = pd.DataFrame(rows)
folds

## E0 · CNN del starter (misma arquitectura y entrenamiento, para comparar en esta máquina)

In [ ]:
mu_x = Xtr[trn].mean((0, 1, 2)); sd_x = Xtr[trn].std((0, 1, 2))
anom_sd = (Ytr[trn] - clim_trn[hour(ttr[trn])]).std((0, 1, 2))
to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=dev)
CLIM, ASD = to_t(clim_trn), to_t(anom_sd)
DEM = to_t((dem - dem.mean()) / dem.std())[None, None]

class StarterNet(nn.Module):
    def __init__(self, w=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
    def forward(self, x, h):
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = interp(self.enc(torch.cat([x, hh], 1)))
        f = torch.cat([f, DEM.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)

def prep(W, t): return to_t((W - mu_x) / sd_x), torch.tensor(hour(t), dtype=torch.float32, device=dev)

def train_loop(model, forward_loss, predict_fn, epochs, idx_fit, lr=2e-3, bs=32, name='', eval_idx=None):
    """Número de épocas fijo; se queda con la última (no se elige época mirando la validación).
    Si se pasa eval_idx, solo se registra el score_ref por época para ver la curva."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    itr = np.array(idx_fit)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * int(np.ceil(len(itr) / bs)))
    hist = []
    for ep in range(epochs):
        model.train(); np.random.shuffle(itr); t0 = time.time()
        for s in range(0, len(itr), bs):
            loss = forward_loss(itr[s:s+bs])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if eval_idx is not None:
            sc = scores(predict_fn(model, eval_idx), Ytr[eval_idx])
            hist.append(sc['score_ref'])
            print(f"{name} época {ep+1:02d} · score_ref val {sc['score_ref']:.4f} · {time.time()-t0:.0f}s")
        else:
            print(f"{name} época {ep+1:02d} · {time.time()-t0:.0f}s")
    return hist

def starter_loss(b):
    x, h = prep(Wtr[b], ttr[b])
    y = (to_t(Ytr[b]) - CLIM[h.long()]) / ASD
    return F.mse_loss(starter(x, h), y)

def starter_predict(model, idx, bs=128):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(idx), bs):
            b = idx[s:s+bs]; x, h = prep(Wtr[b], ttr[b])
            out.append((CLIM[h.long()] + model(x, h) * ASD).cpu().numpy())
    return np.concatenate(out)

torch.manual_seed(SEED); np.random.seed(SEED)
starter = StarterNet().to(dev)
t0 = time.time()
hist_starter = train_loop(starter, starter_loss, starter_predict, EP_STARTER, np.where(trn)[0], name='starter', eval_idx=np.where(val)[0])
print('tiempo total: %.1f min' % ((time.time() - t0) / 60))
pred_starter = starter_predict(starter, np.where(val)[0])
log('E0e CNN del starter', pred_starter)

## E1 · Residuo sobre GFS en la grilla de 3 km

- Entradas (por celda de 3 km): GFS t−2, t−1, t interpolado y normalizado (12 canales), relieve, diferencia con el relieve suavizado, pendiente, hora en seno/coseno y un mapa estático aprendido.
- Salida: corrección `WRF − GFS interpolado(t)` para las 4 variables.
- Pérdida: error cuadrático de cada variable dividido por el RMSE de referencia de la climatología, igual que pesa en la métrica.
- No usa día del año, para no depender de la estación.
- Todo lo que el modelo necesita para predecir (normalización, relieve, grilla de interpolación) vive dentro del modelo como buffers, así que el `state_dict` es autosuficiente.
- En modo evaluación la RH se recorta a [0, 100].

In [ ]:
res_sd = (Ytr[trn] - Xi_tr[trn]).std((0, 1, 2)).astype(np.float32)
REFt = to_t(REF)
norm = lambda a: (a - a.mean()) / a.std()
STATIC = to_t(np.stack([norm(dem), norm(dz), norm(slope)]))[None]    # (1,3,70,35)
LOSS = os.environ.get('LOSS', 'mse')

class Block(nn.Module):
    def __init__(self, w):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(w, w, 3, padding=1), nn.Conv2d(w, w, 3, padding=1)
    def forward(self, x): return x + self.c2(F.gelu(self.c1(F.gelu(x))))

class ResidualNet(nn.Module):
    def __init__(self, w, n_blocks, grid, mu, sd, rsd, static):
        super().__init__()
        # buffers: quedan dentro del state_dict, así la app no depende de variables globales del notebook
        self.register_buffer('grid', grid.clone())      # (1,70,35,2) coordenadas de muestreo del GFS
        self.register_buffer('mu', mu.clone())          # (1,4,1,1) media de entrada (GFS, unidades físicas)
        self.register_buffer('sd', sd.clone())          # (1,4,1,1)
        self.register_buffer('rsd', rsd.clone())        # (1,4,1,1) desviación del residuo WRF - GFS
        self.register_buffer('rel', static.clone())     # (1,3,70,35) relieve, dif. con relieve suavizado, pendiente
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.inp = nn.Conv2d(12 + 3 + 2 + 8, w, 3, padding=1)
        self.blocks = nn.Sequential(*[Block(w) for _ in range(n_blocks)])
        self.out = nn.Conv2d(w, 4, 1)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)   # arranca prediciendo el GFS interpolado
    def _interp(self, x):
        return F.grid_sample(x, self.grid.expand(x.shape[0], -1, -1, -1), mode='bilinear', align_corners=True)
    def forward(self, w_raw, h):
        # w_raw: (B,3,17,13,4) en unidades físicas (RH %, T K, U y V m/s), GFS fila 0 = norte; h: (B,) hora UTC
        B = w_raw.shape[0]
        frames = w_raw.permute(0, 1, 4, 2, 3)                          # (B,3,4,17,13)
        base = self._interp(frames[:, 2])                              # GFS(t) interpolado, unidades físicas (B,4,70,35)
        xin = self._interp(((frames - self.mu[:, None]) / self.sd[:, None]).reshape(B, 12, 17, 13))
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 70, 35)
        z = torch.cat([xin, self.rel.expand(B, -1, -1, -1), hh, self.static.expand(B, -1, -1, -1)], 1)
        out = base + self.out(self.blocks(self.inp(z))) * self.rsd
        if not self.training:                                          # RH física en [0, 100]
            out = torch.cat([out[:, :1].clamp(0, 100), out[:, 1:]], 1)
        return out.permute(0, 2, 3, 1)                                 # predicción WRF (B,70,35,4), fila 0 = sur

W_E1, NB_E1 = 48, 6
def build_resnet(mu, sd, rsd):
    v = lambda a: to_t(a).view(1, 4, 1, 1)
    return ResidualNet(W_E1, NB_E1, GRID, v(mu), v(sd), v(rsd), STATIC).to(dev)

def res_loss_fn(model):
    def f(b):
        w, h = to_t(Wtr[b]), torch.tensor(hour(ttr[b]), dtype=torch.float32, device=dev)
        err2 = ((model(w, h) - to_t(Ytr[b])) ** 2).mean((0, 1, 2))
        if LOSS == 'rmse':                                              # alineada con la métrica (promedio de RMSE_v / REF_v)
            return (torch.sqrt(err2 + 1e-6) / REFt).mean()
        return (err2 / REFt ** 2).mean()
    return f

def res_predict(model, idx, W=None, t=None, bs=64):
    W = Wtr if W is None else W; t = ttr if t is None else t
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(idx), bs):
            b = idx[s:s+bs]
            out.append(model(to_t(W[b]), torch.tensor(hour(t[b]), dtype=torch.float32, device=dev)).cpu().numpy())
    return np.concatenate(out)

# Modelo de validación: entrena solo con trn (antes del 16 jun) para medir; el modelo final se reentrena más abajo
torch.manual_seed(SEED); np.random.seed(SEED)
resnet = build_resnet(mu_x, sd_x, res_sd)
print('pérdida:', LOSS, '| parámetros E1: %.2f M' % (sum(p.numel() for p in resnet.parameters()) / 1e6))
t0 = time.time()
hist_res = train_loop(resnet, res_loss_fn(resnet), res_predict, EP_E1, np.where(trn)[0], name='E1', eval_idx=np.where(val)[0])
print('tiempo total: %.1f min' % ((time.time() - t0) / 60))
pred_res = res_predict(resnet, np.where(val)[0])
log('E1 residuo sobre GFS (CNN 3 km)', pred_res)


## Resumen

In [ ]:
tabla = pd.DataFrame(RESULTS).set_index('experimento').round(4)
os.makedirs('results', exist_ok=True)
tabla.to_csv(f'results/exp01_resultados{TAG}.csv')
print(tabla.to_string())

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
tabla['score_local'].plot.barh(ax=ax[0], color='#4c78a8'); ax[0].set_title('Score local (validación desde el 16 jun)'); ax[0].invert_yaxis()
(tabla[[f'RMSE_{v}' for v in VARS]] / REF).plot.barh(ax=ax[1]); ax[1].set_title('RMSE / RMSE de referencia por variable (menor es mejor)'); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
ax[0].plot(hist_starter, label='CNN del starter'); ax[0].plot(hist_res, label='E1 residuo'); ax[0].set_title('score_ref de validación por época'); ax[0].legend()
i = np.where(val)[0][np.where(ttr[val] == '2024062018')[0][0]] if (ttr[val] == '2024062018').any() else np.where(val)[0][0]
j = list(np.where(val)[0]).index(i)
vmin, vmax = Ytr[i, ..., 1].min() - 273.15, Ytr[i, ..., 1].max() - 273.15
ax[1].imshow(Ytr[i, ..., 1] - 273.15, origin='lower', cmap='coolwarm', vmin=vmin, vmax=vmax); ax[1].set_title(f'WRF T (°C) {ttr[i]}')
im = ax[2].imshow(pred_res[j, ..., 1] - 273.15, origin='lower', cmap='coolwarm', vmin=vmin, vmax=vmax); ax[2].set_title('E1 T (°C)')
plt.colorbar(im, ax=ax[2], shrink=0.8); plt.tight_layout(); plt.show()

## Modelo final: reentrenamiento con todos los datos

Con el número de épocas fijado arriba (sin elegir nada con la validación), se reentrena E1 con todo el train, incluidas las dos últimas semanas de junio, que son las más parecidas al periodo de test. La normalización (`mu`, `sd`, desviación del residuo) se recalcula con todos los datos. Este es el modelo que se guarda y el que genera el submission.

In [ ]:
ALL = np.arange(len(ttr))
mu_a, sd_a = Xtr.mean((0, 1, 2)), Xtr.std((0, 1, 2))
res_a = (Ytr - Xi_tr).std((0, 1, 2)).astype(np.float32)
torch.manual_seed(SEED); np.random.seed(SEED)
final = build_resnet(mu_a, sd_a, res_a)
t0 = time.time()
train_loop(final, res_loss_fn(final), res_predict, EP_E1, ALL, name='E1 final (todo el train)')
print('tiempo total: %.1f min' % ((time.time() - t0) / 60))


## Guardado (contrato con la app)

Solo tensores, listas, números y texto, para que `torch.load` funcione con la configuración por defecto de torch 2.6 o posterior. El archivo queda en `models/` y no se sube a GitHub (`.gitignore` excluye `*.pt`, salvo `models/modelo_final.pt`). Después se recarga el archivo desde cero y se comprueba que da las mismas predicciones.

In [ ]:
mdir = '../models' if os.path.isdir('../models') else 'models'
os.makedirs(mdir, exist_ok=True)
ruta = os.path.join(mdir, f'exp01_residuo{TAG}.pt')
torch.save({
    'state_dict': final.state_dict(),                     # incluye grilla de interpolación, normalización y relieve (buffers)
    'config': {'w': W_E1, 'n_blocks': NB_E1},
    'gfs_lats': torch.tensor(glat), 'gfs_lons': torch.tensor(glon),
    'wrf_lats': torch.tensor(wlat), 'wrf_lons': torch.tensor(wlon), 'dem': torch.tensor(dem),
    'epocas': EP_E1, 'perdida': LOSS, 'datos_entrenamiento': 'train completo (18 feb a 30 jun 2024)',
    'orden_variables': VARS, 'unidades_entrada': ['%', 'K', 'm/s', 'm/s'],
    'hora': 'UTC (0 a 23); la app trabaja en hora de Perú, convertir antes de llamar al modelo',
    'orden_filas_entrada': 'GFS: fila 0 = norte, columna 0 = oeste',
    'orden_filas_salida': 'WRF: fila 0 = sur, columna 0 = oeste',
    'ventanas': '(N,3,17,13,4) con t-2, t-1, t; si falta un retraso se usa el disponible más cercano a t (t-1 antes que t)',
    'salida': 'WRF (N,70,35,4) en unidades originales; RH recortada a [0,100] en modo eval',
}, ruta)
print('guardado en', ruta, '(%.1f MB)' % (os.path.getsize(ruta) / 1e6))


In [ ]:
# Comprobación del contrato: se reconstruye el modelo solo con el archivo y debe coincidir con el original
ck = torch.load(ruta, map_location=dev)          # configuración por defecto (weights_only=True)
ck_m = ResidualNet(ck['config']['w'], ck['config']['n_blocks'], torch.zeros(1, 70, 35, 2), torch.zeros(1, 4, 1, 1),
                   torch.ones(1, 4, 1, 1), torch.ones(1, 4, 1, 1), torch.zeros(1, 3, 70, 35)).to(dev)
ck_m.load_state_dict(ck['state_dict'])
chk = np.where(val)[0][:16]
dif = np.abs(res_predict(final, chk) - res_predict(ck_m, chk)).max()
print('diferencia máxima entre el modelo original y el recargado:', dif)
assert dif < 1e-4


## Submission de prueba (opcional)
Genera `results/exp01_submission.csv` con el modelo final E1 (reentrenado con todo el train) en el formato de Kaggle, por si se quiere enviar para comparar con el leaderboard público.

In [ ]:
sample = pd.read_csv(find('sample_submission.csv'))
P = res_predict(final, np.arange(len(Xte)), W=Wte, t=tte)
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti, r, c = parts[0].map(pos).values, parts[1].astype(int).values, parts[2].astype(int).values
sub = pd.DataFrame({'id': sample['id']})
for kk, v in enumerate(VARS): sub[v] = P[ti, r, c, kk].round(3)
assert sub.shape == sample.shape and np.isfinite(sub[VARS].values).all()
sub.to_csv(f'results/exp01_submission{TAG}.csv', index=False)
print(sub.shape)